# Web Scraping

## Billboard Scraping

The first step is to scrape the Billboard Global 200 chart for a given week.

In [1]:
#Import necessary packages
import time
import requests
import lxml.html as lx
import re
import pandas as pd
from datetime import datetime
import sqlite3 as sql
from requests.exceptions import HTTPError

In [2]:
#Set up base url and headers for scraping
base_url = 'https://www.billboard.com/charts'
headers = {
    'User-Agent': "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/26.0.1 Safari/605.1.15"
}

In [3]:
# Function that retrieves and returns the HTML of a website
# If no website is found, return None
def get_html(link):
    try:
        billboard_info = requests.get(link, headers = headers)
        billboard_info.raise_for_status()
        billboard_html = lx.fromstring(billboard_info.text)
        return billboard_html
    except HTTPError as err:
        return None

In [5]:
# For the get_billboard_date(), this function cleans the artist name and song string; remove unnecessary characters
def clean_string(stc):
    new_string = re.findall(r'(\S.*\S)', stc)
    if len(new_string) != 0:
        return new_string[0]
    else:
        return None

In [6]:
# For the get_billboard_data(), this function cleans the billboard position/peak weak/etc. string
def clean_int(stc):
    new_int = re.sub(r'\s*', '', stc)
    return new_int

In [7]:
# For the get_billboard_data(), this function cleans the billboard week date
def clean_date(stc):
    str_date = re.findall(r"(?<=of )(.*)(?=\s)", stc)
    if len(str_date) != 0:
        str_date = str_date[0]
        new_date_format = datetime.strptime(str_date, "%B %d, %Y").date()
        reformatted_str_date = new_date_format.strftime("%m-%d-%Y")
        return reformatted_str_date
    else:
        return None

In [8]:
# This function parses through the Billboard website HTML and returns a data frame containing the following columns for a specific row in the Billboard Chart:
# - Song Rank
# - Song name
# - Artist name(s)
# - Artist link(s)
# - Last week rank
# - Song Peak Position
# - Weeks in Chart
# - Billboard Chart week
def get_billboard_data(billboard_html, row):

    billboard_row = {}
    billboard_row['rank'] = row
    billboard_row['song_name'] = clean_string(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//h3/text()")[0])

    len_artist = len(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a"))
    if len_artist == 0:
        billboard_row['artist'] = clean_string(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//li[@class='o-chart-results-list__item // lrv-u-flex-grow-1 lrv-u-flex lrv-u-flex-direction-column lrv-u-justify-content-center lrv-u-padding-l-050 lrv-u-padding-l-00@mobile-max u-max-width-397']//span/text()")[0])
        billboard_row['artist_link'] = None
    else:
        artists = billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/text()")[0]
        links = billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/@href")[0]
        for i in range(1, len_artist):
            artists = artists + ', ' + billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/text()")[i]
            links = links + ', ' + billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/@href")[i]
        billboard_row['artist'] = artists
        billboard_row['artist_link'] = links
    billboard_row['last_week_rank'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[0].text_content())
    billboard_row['peak'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[1].text_content())
    billboard_row['weeks_chart'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[2].text_content())
    billboard_row['week'] = clean_date(billboard_html.xpath("//div[@class='charts-title lrv-u-position-relative // lrv-u-width-100p u-padding-t-2.25 u-margin-b-0.625@tablet lrv-u-padding-t-050@mobile-max']//div[@class = 'u-padding-a-1@mobile-max u-padding-b-0.375@mobile-max']")[0].text_content())

    row_df = pd.DataFrame(billboard_row, index = [billboard_row["rank"]])
    return row_df

In [9]:
# This function takes a link as input and retrieves/returns a data frame with all the Billboard data for every row in the Billboard Chart
def get_billboard_chart(link):

    billboard_rows = []

    billboard_html = get_html(link)
    num_of_elements = len(billboard_html.xpath("//div[@class='o-chart-results-list-row-container']"))

    for i in range(1, num_of_elements + 1):
        new_row = get_billboard_data(billboard_html, i)
        billboard_rows.append(new_row)

    complete_chart = pd.concat(billboard_rows)

    return complete_chart

In [10]:
# This function takes the link to the Billboard Charts homepage as input and returns a dictionary containing the link to all the following Billboard Charts:
# - Global 200 Songs
# - US Hot 100
# - All Billboard Charts for individual countries
# In the dictionary, the chart name is the key and the chart link is the stored value.
def get_billboard_chart_links(link):
    billboard_html = get_html(link)
    base_direc = 'https://www.billboard.com'

    charts_dic = {}
    chart_categories = billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-hits-of-the-world']//a/text()")
    chart_links = billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-hits-of-the-world']//a/@href")

    #US table
    hot_100_link = chart_links.append(billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-top-charts']//a/@href")[0])
    chart_categories.append('U.S. Hot 100 Songs')

    #Global 200
    hot_200_link = chart_links.append(billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-global']//a/@href")[0])
    chart_categories.append('Worldwide Top 200 Songs')

    for element, element_link in zip(chart_categories, chart_links):
        charts_dic[clean_string(element)] = base_direc + element_link

    return charts_dic

In [235]:
# This function filters through the Billboard charts for countries and removes duplicate charts for the same country
# Then, from the adjusted list, get the Billboard data for that chart using the get_billboard_chart() function.
# To each data frame, add a new column that stores the name of the chart
# Save EACH chart data frame in a dictionary called all_charts.
# Return the dictionary all_charts
def get_all_billboard_global_charts(url):
    world_chart_links = get_billboard_chart_links(url)
    adjusted_world_charts = [chart_name for chart_name in list(world_chart_links.keys()) if re.search("Artist|Album|Global|Top Philippine|Thai Country|Vietnamese|Singles", chart_name) is None]

    all_charts = {}

    for chart in adjusted_world_charts:
        time.sleep(1)
        #print(chart)
        chart_link = world_chart_links[chart]
        chart_table = get_billboard_chart(chart_link)
        column_add = [chart] * chart_table.shape[0]
        chart_table.insert(loc = chart_table.shape[1], column = "chart", value = column_add)
        all_charts[chart] = chart_table

    return all_charts

In [236]:
all_charts_list_df = get_all_billboard_global_charts(base_url)

## Data Processing (Billboard)

In [237]:
#From the dictionary created using the get_all_billboard_global_charts() function, create one data frame called all_charts_df by concating the data frames in the dictionary.
all_charts_df = pd.concat(all_charts_list_df).reset_index(drop=True)
all_charts_df = all_charts_df.reindex(columns=["artist", "song_name", "rank", "artist_link", "last_week_rank", "peak", "weeks_chart", "week", "chart"])

In [238]:
all_charts_df

,artist,song_name,rank,artist_link,last_week_rank,peak,weeks_chart,week,chart
0,Flipperachi,FA9LA,1,None,1,1,10,02-28-2026,Billboard Arabic Hot 100
1,DYSTINCT,Yama,2,None,2,1,17,02-28-2026,Billboard Arabic Hot 100
2,Vanco Featuring AYA,Ma Tnsani,3,None,3,2,29,02-28-2026,Billboard Arabic Hot 100
3,"Lege-Cy, Ghaliaa",Msh Awl Marra,4,None,7,4,3,02-28-2026,Billboard Arabic Hot 100
4,DYSTINCT,Ta3al,5,None,4,3,4,02-28-2026,Billboard Arabic Hot 100
...,...,...,...,...,...,...,...,...,...
1645,Don Omar,Danza Kuduro,196,https://www.billboard.com/artist/don-omar/,-,140,37,02-28-2026,Worldwide Top 200 Songs
1646,Jimin,Who,197,https://www.billboard.com/artist/jimin/,-,1,74,02-28-2026,Worldwide Top 200 Songs
1647,Hozier,Too Sweet,198,https://www.billboard.com/artist/hozier/,-,1,98,02-28-2026,Worldwide Top 200 Songs
1648,"Shakira, Wyclef Jean",Hips Don't Lie,199,"https://www.billboard.com/artist/shakira/, htt...",-,134,20,02-28-2026,Worldwide Top 200 Songs


In [241]:
#Total number of mentions of artist in charts (does not account for collaboration)
def artist_nonunique_mentions(df):
    total_appearances_all_charts = df.groupby("artist").count().sort_values(by=['song_name'], ascending = False)[['song_name', 'week']]
    total_appearances_all_charts['week'] = df.loc[0, 'week']
    total_appearances_all_charts  = total_appearances_all_charts.rename(columns = {"song_name" : "count"}).reset_index()
    return total_appearances_all_charts

In [240]:
total_appearances_all_charts = artist_nonunique_mentions(all_charts_df)

In [242]:
def unique_artist_list(df):
    all_artists_combination = [str(entry).lower() for entry in total_appearances_all_charts['artist'].unique()]
    unique_artists = []
    for entry in all_artists_combination:
        entry_adjusted = re.sub(r'(\sx\s)|(feat.)|(\bft.\b)|(featuring)|(with)|(&)|(\+)', ', ', entry)
        all_artists_in_entry = re.split(r'[,]', entry_adjusted)
        for artist in all_artists_in_entry:
            no_whitespace = re.findall(r'(\w.*\w)', artist)
            if no_whitespace not in unique_artists and len(no_whitespace) >= 1:
                unique_artists.append(no_whitespace[0])
    return unique_artists

In [243]:
def artist_mentions_unique(df):
    unique_artists = unique_artist_list(df)

    individual_artist_chart_appearances_dict = {}
    for artist in unique_artists:
        artist_count = total_appearances_all_charts[total_appearances_all_charts['artist'].str.lower().str.contains(artist, regex = False)].sum(axis = 0).iloc[1]
        individual_artist_chart_appearances_dict[artist] = artist_count

    individual_artist_chart_appearances_df = pd.Series(individual_artist_chart_appearances_dict).to_frame().reset_index().rename(columns= {'index' : 'artist', 0 : 'count'})
    week = df.iloc[0, 2]
    individual_artist_chart_appearances_df['week'] = [week] * individual_artist_chart_appearances_df.shape[0]
    return individual_artist_chart_appearances_df

In [244]:
unique_artist_mentions = artist_mentions_unique(total_appearances_all_charts)

In [245]:
#How many charts each artist is charting in
def artist_chart_count(df):
    unique_chart_appearances = df.groupby(by = ["artist", "chart"]).count().groupby("artist").count().iloc[:, 0].sort_values(ascending=False).to_frame()
    unique_chart_appearances['week'] = df.loc[0, 'week']
    unique_chart_appearances = unique_chart_appearances.rename(columns = {"song_name" : "unique_count"}).reset_index()
    week = df.iloc[0, 7]
    unique_chart_appearances['week'] = [week] * unique_chart_appearances.shape[0]
    return unique_chart_appearances

In [246]:
unique_chart_appearances = artist_chart_count(all_charts_df)

In [247]:
total_appearances_all_charts

,artist,count,week
0,Bad Bunny,180,02-28-2026
1,Olivia Dean,62,02-28-2026
2,sombr,42,02-28-2026
3,Taylor Swift,37,02-28-2026
4,Zara Larsson,25,02-28-2026
...,...,...,...
697,Jade LeMac,1,02-28-2026
698,Jamie Fine,1,02-28-2026
699,Jane Zhang,1,02-28-2026
700,Jason Aldean,1,02-28-2026


In [248]:
unique_artist_mentions

,artist,count,week
0,bad bunny,199,02-28-2026
1,olivia dean,62,02-28-2026
2,sombr,42,02-28-2026
3,taylor swift,37,02-28-2026
4,zara larsson,35,02-28-2026
...,...,...,...
909,jade lemac,1,02-28-2026
910,jamie fine,1,02-28-2026
911,jane zhang,1,02-28-2026
912,jason aldean,1,02-28-2026


In [249]:
unique_chart_appearances

,artist,unique_count,week
0,Bad Bunny,39,02-28-2026
1,Taylor Swift,28,02-28-2026
2,Djo,25,02-28-2026
3,"Dave, Tems",24,02-28-2026
4,RAYE,24,02-28-2026
...,...,...,...
697,"J. Cole, Tems, Future",1,02-28-2026
698,JC Reyes & Slayter,1,02-28-2026
699,"Jackie Chan, Emil Wakin Chau, Leehom Wang, Nic...",1,02-28-2026
700,Jacub,1,02-28-2026


## Streaming Scraping

In [250]:
spotify_url = 'https://kworb.net/spotify/'
apple_url = 'https://kworb.net'

In [251]:
def get_spotify_streaming_links(base_url_spotify):
    streaming_spotify_html = get_html('https://kworb.net/spotify/')
    charts_spotify_href = streaming_spotify_html.xpath("//table[@style='width: 410px;']//tr//@href")
    spotify_daily_charts_url = [(base_url_spotify + entry) for entry in charts_spotify_href if (not re.search('totals', entry)) and re.search('daily', entry)]
    spotify_weekly_charts_url = [(base_url_spotify + entry) for entry in charts_spotify_href if (not re.search('totals', entry)) and re.search('weekly', entry)]
    return spotify_daily_charts_url, spotify_weekly_charts_url

In [252]:
spotify_daily_links, spotify_weekly_links = get_spotify_streaming_links(spotify_url)

In [253]:
def get_apple_streaming_links(base_url_apple):
    streaming_apple_html = get_html('https://kworb.net/charts/')
    charts_apple_href = streaming_apple_html.xpath("//table[@style='width:1000px']//tr//@href")
    base_url_apple = 'https://kworb.net'
    apple_daily_charts_url = [(base_url_apple + entry) for entry in charts_apple_href if re.search('apple', entry)]
    return apple_daily_charts_url

In [254]:
apple_daily_links = get_apple_streaming_links(apple_url)

In [255]:
def get_streaming_charts_spotify(link_list):

    all_country_streaming_charts = []
    for link in link_list:
        #print(link)
        #time.sleep(1)
        chart_html = get_html(link)
        if chart_html == None:
            continue
        table = pd.read_html(link, storage_options=headers)[0]
        table_name_content = chart_html.xpath("//div//span[@class='pagetitle']")[0].text_content()
        table_name_mid = re.findall(r'\S.*-', table_name_content)
        table_name = re.sub(r"- |-", '', table_name_mid[0])

        chart_date_info = chart_html.xpath("//div//span[@class='pagetitle']")[0].text_content()
        chart_date = re.findall(r'(\d.*\d)', chart_date_info)
        new_date_format = datetime.strptime(chart_date[0], "%Y/%m/%d").date()
        reformatted_str_date = new_date_format.strftime("%m-%d-%Y")

        table['chart_name'] = [table_name] * table.shape[0]
        table['date'] = [reformatted_str_date] * table.shape[0]
        all_country_streaming_charts.append(table)

    all_country_streaming_charts_df = pd.concat(all_country_streaming_charts)

    return all_country_streaming_charts_df

In [256]:
def get_charts_apple(link_list):
    all_country_streaming_charts = []
    for link in link_list:
        #print(link)
        #time.sleep(1)

        table = pd.read_html(link, storage_options=headers)[0]

        chart_html = get_html(link)
        if chart_html == None:
            continue
        table_name_content = chart_html.xpath("//div//span[@class='pagetitle']")[0].text_content()
        table_name = re.findall(r'(\S.*Song)', table_name_content)[0]
        reformatted_str_date = datetime.now().strftime("%m-%d-%Y")

        table['chart_name'] = [table_name] * table.shape[0]
        table['date'] = [reformatted_str_date] * table.shape[0]
        all_country_streaming_charts.append(table)

    all_country_streaming_charts_df = pd.concat(all_country_streaming_charts)
    return all_country_streaming_charts_df

In [37]:
spotify_daily_streaming_charts = get_streaming_charts_spotify(spotify_daily_links)

In [46]:
spotify_weekly_streaming_charts = get_streaming_charts_spotify(spotify_weekly_links)

In [63]:
apple_charts = get_charts_apple(apple_daily_links)

https://kworb.net/apple_songs/index.html
https://kworb.net/charts/apple_s/us.html
https://kworb.net/charts/apple_s/uk.html
https://kworb.net/charts/apple_s/ar.html
https://kworb.net/charts/apple_s/au.html
https://kworb.net/charts/apple_s/br.html
https://kworb.net/charts/apple_s/ca.html
https://kworb.net/charts/apple_s/co.html
https://kworb.net/charts/apple_s/fr.html
https://kworb.net/charts/apple_s/de.html
https://kworb.net/charts/apple_s/in.html
https://kworb.net/charts/apple_s/id.html
https://kworb.net/charts/apple_s/it.html
https://kworb.net/charts/apple_s/jp.html
https://kworb.net/charts/apple_s/mx.html
https://kworb.net/charts/apple_s/nl.html
https://kworb.net/charts/apple_s/pe.html
https://kworb.net/charts/apple_s/ph.html
https://kworb.net/charts/apple_s/pl.html
https://kworb.net/charts/apple_s/ru.html
https://kworb.net/charts/apple_s/kr.html
https://kworb.net/charts/apple_s/es.html
https://kworb.net/charts/apple_s/tw.html
https://kworb.net/charts/apple_s/th.html
https://kworb.ne

## Data Processing (Streaming)

In [259]:
def clean_spotify(all_country_streaming_charts_df):
    title = []
    artist = []
    country = []
    for entry in all_country_streaming_charts_df["Artist and Title"]:
        new_entry = re.split(r' - ', entry)
        artist.append(new_entry[0])
        title.append(new_entry[1])
    for entry in all_country_streaming_charts_df['chart_name']:
        new_entry = re.findall(r'(?<=Chart )(\w.*\w)', entry)
        country.append(new_entry[0])
    all_country_streaming_charts_df['artist'] = artist
    all_country_streaming_charts_df['song_name'] = title
    all_country_streaming_charts_df['country'] = country

    all_country_streaming_charts_df = all_country_streaming_charts_df.rename(columns = {'Pos': "rank", "Pk": "peak"})

    all_country_streaming_charts_df = all_country_streaming_charts_df.reindex(columns=["rank", "artist", "song_name", "Days", "peak", "Streams", "Streams+", "7Day", "7Day+", "Total", "country", "chart_name", "date"])
    return all_country_streaming_charts_df

In [295]:
def clean_spotify_weekly(all_country_streaming_charts_df):
    title = []
    artist = []
    country = []
    for entry in all_country_streaming_charts_df["Artist and Title"]:
        new_entry = re.split(r' - ', entry)
        artist.append(new_entry[0])
        title.append(new_entry[1])
    for entry in all_country_streaming_charts_df['chart_name']:
        new_entry = re.findall(r'(?<=Chart )(\w.*\w)', entry)
        country.append(new_entry[0])
    all_country_streaming_charts_df['artist'] = artist
    all_country_streaming_charts_df['song_name'] = title
    all_country_streaming_charts_df['country'] = country

    all_country_streaming_charts_df = all_country_streaming_charts_df.rename(columns = {'Pos': "rank", "Pk": "peak", "Wks": "weeks"})

    all_country_streaming_charts_df = all_country_streaming_charts_df.reindex(columns=["rank", "artist", "song_name", "peak", "weeks", "Streams", "Streams+", "Total", "country", "chart_name", "date"])
    return all_country_streaming_charts_df

In [296]:
def clean_apple_charts(all_country_streaming_charts_df):
    title = []
    artist = []
    country = []

    for entry in range(all_country_streaming_charts_df.shape[0]):
        if type(all_country_streaming_charts_df.iloc[entry, 15]) == float:
            temp_entry = all_country_streaming_charts_df.iloc[entry, 2]
            new_entry = re.split(r' - ', temp_entry)
            artist.append(new_entry[0])
            title.append(new_entry[1])
        else:
            temp_entry = all_country_streaming_charts_df.iloc[entry, 15]
            new_entry = re.split(r' - ', temp_entry)
            artist.append(new_entry[0])
            title.append(new_entry[1])
    for entry in all_country_streaming_charts_df['chart_name']:
        new_entry = re.findall(r'(\w.*\w)(?= Apple)', entry)
        country.append(new_entry[0])
    all_country_streaming_charts_df['artist'] = artist
    all_country_streaming_charts_df['song_name'] = title
    all_country_streaming_charts_df['country'] = country

    all_country_streaming_charts_df = all_country_streaming_charts_df.rename(columns = {'Pos': "rank"})

    all_country_streaming_charts_df = all_country_streaming_charts_df.reindex(columns=["rank", "artist", "song_name", "country", "chart_name", "date"])
    return all_country_streaming_charts_df

In [ ]:
spotify_daily_streaming_charts = clean_spotify(spotify_daily_streaming_charts)

In [ ]:
spotify_weekly_streaming_charts = clean_spotify_weekly(spotify_weekly_streaming_charts)

In [ ]:
apple_charts = clean_apple_charts(apple_charts)

In [308]:
spotify_daily_streaming_charts

,rank,artist,song_name,Days,peak,Streams,Streams+,7Day,7Day+,Total,chart_name,date
0,1,Bad Bunny,DtMF,418,1,6182642,-111492.0,45722046,-917832,1549114511,Spotify Daily Chart Global,02-26-2026
1,2,PinkPantheress,Stateside + Zara Larsson (w/ Zara Larsson),51,2,5230758,284294.0,25677341,2827496,122108411,Spotify Daily Chart Global,02-26-2026
2,3,Djo,End of Beginning,741,1,4856294,-161385.0,32979053,-19009,2223880869,Spotify Daily Chart Global,02-26-2026
3,4,Taylor Swift,The Fate of Ophelia,147,1,4545484,20750.0,30757410,104091,1067922884,Spotify Daily Chart Global,02-26-2026
4,5,Bad Bunny,BAILE INoLVIDABLE,418,2,4536650,4256.0,33278386,-751104,1273769628,Spotify Daily Chart Global,02-26-2026
...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,Đen,Đi Về Nhà (w/ JustaTee),1660,1,21236,-1850.0,193679,-33738,37789777,Spotify Daily Chart Vietnam,02-26-2026
196,197,Dhruv,double take,1472,7,21082,334.0,41830,21082,32031237,Spotify Daily Chart Vietnam,02-26-2026
197,198,Vũ Cát Tường,Từng Là,678,2,21032,NaN,21032,693,38007812,Spotify Daily Chart Vietnam,02-26-2026
198,199,MONO,Chăm Hoa,483,6,21016,NaN,103427,2105,24670401,Spotify Daily Chart Vietnam,02-26-2026


In [301]:
spotify_weekly_streaming_charts

,rank,artist,song_name,peak,weeks,Streams,Streams+,Total,chart_name,date
0,1,Bad Bunny,DtMF,1,60,45722046,-14850248.0,1549114511,Spotify Weekly Chart Global,02-26-2026
1,2,Bad Bunny,BAILE INoLVIDABLE,2,60,33278386,-10608661.0,1273769628,Spotify Weekly Chart Global,02-26-2026
2,3,Djo,End of Beginning,1,106,32979053,-2245987.0,2225140553,Spotify Weekly Chart Global,02-26-2026
3,4,Bad Bunny,NUEVAYoL,3,60,32180613,-10955089.0,1040380429,Spotify Weekly Chart Global,02-26-2026
4,5,Olivia Dean,Man I Need,3,27,31180371,-1623349.0,792309416,Spotify Weekly Chart Global,02-26-2026
...,...,...,...,...,...,...,...,...,...,...
195,196,Changg,Em Không Hiểu (w/ Minh Huy),27,237,138341,-10117.0,32312825,Spotify Weekly Chart Vietnam,02-26-2026
196,197,Lil Zpoet,Yêu Từ Đâu Mà Ra,150,20,137045,-1849.0,3332279,Spotify Weekly Chart Vietnam,02-26-2026
197,198,Sơn Tùng M-TP,Chúng Ta Không Thuộc Về Nhau,60,60,137040,NaN,8636859,Spotify Weekly Chart Vietnam,02-26-2026
198,199,AMEE,MỘNG YU (w/ RPT MCK),3,76,136984,NaN,25553883,Spotify Weekly Chart Vietnam,02-26-2026


In [302]:
apple_charts

,rank,artist,song_name,chart_name,date
0,1,Bad Bunny,DtMF,Worldwide Apple Music Song,02-27-2026
1,2,Taylor Swift,The Fate of Ophelia,Worldwide Apple Music Song,02-27-2026
2,3,PinkPantheress,Stateside,Worldwide Apple Music Song,02-27-2026
3,4,Dave & Tems,Raindance,Worldwide Apple Music Song,02-27-2026
4,5,Olivia Dean,Man I Need,Worldwide Apple Music Song,02-27-2026
...,...,...,...,...,...
195,196,Jnr Spragga,Zvakufaya (feat. Nisha Ts),Zimbabwe Apple Music Top Song,02-27-2026
196,197,Jah Prayzah,Shuga,Zimbabwe Apple Music Top Song,02-27-2026
197,198,"Mr Pilato, Ego Slimflow & Tebogo G Mashego","Biri Marung (feat. Sje Konka, Focalistic, DJ M...",Zimbabwe Apple Music Top Song,02-27-2026
198,199,The Silent Partner,Im Learning (feat. IVEY.H),Zimbabwe Apple Music Top Song,02-27-2026


In [309]:
apple_get_country = []
for entry in apple_charts['chart_name']:
    new_entry = re.findall(r'(\w.*\w)(?= Apple)', entry)
    apple_get_country.append(new_entry[0])
apple_charts['country'] = apple_get_country

In [310]:
apple_charts

,rank,artist,song_name,chart_name,date,country
0,1,Bad Bunny,DtMF,Worldwide Apple Music Song,02-27-2026,Worldwide
1,2,Taylor Swift,The Fate of Ophelia,Worldwide Apple Music Song,02-27-2026,Worldwide
2,3,PinkPantheress,Stateside,Worldwide Apple Music Song,02-27-2026,Worldwide
3,4,Dave & Tems,Raindance,Worldwide Apple Music Song,02-27-2026,Worldwide
4,5,Olivia Dean,Man I Need,Worldwide Apple Music Song,02-27-2026,Worldwide
...,...,...,...,...,...,...
195,196,Jnr Spragga,Zvakufaya (feat. Nisha Ts),Zimbabwe Apple Music Top Song,02-27-2026,Zimbabwe
196,197,Jah Prayzah,Shuga,Zimbabwe Apple Music Top Song,02-27-2026,Zimbabwe
197,198,"Mr Pilato, Ego Slimflow & Tebogo G Mashego","Biri Marung (feat. Sje Konka, Focalistic, DJ M...",Zimbabwe Apple Music Top Song,02-27-2026,Zimbabwe
198,199,The Silent Partner,Im Learning (feat. IVEY.H),Zimbabwe Apple Music Top Song,02-27-2026,Zimbabwe


In [311]:
spotify_daily_get_country = []
for entry in spotify_daily_streaming_charts['chart_name']:
    new_entry = re.findall(r'(?<=Chart )(\w.*\w)', entry)
    spotify_daily_get_country.append(new_entry[0])
spotify_daily_streaming_charts['country'] = spotify_daily_get_country

In [313]:
spotify_daily_streaming_charts

,rank,artist,song_name,Days,peak,Streams,Streams+,7Day,7Day+,Total,chart_name,date,country
0,1,Bad Bunny,DtMF,418,1,6182642,-111492.0,45722046,-917832,1549114511,Spotify Daily Chart Global,02-26-2026,Global
1,2,PinkPantheress,Stateside + Zara Larsson (w/ Zara Larsson),51,2,5230758,284294.0,25677341,2827496,122108411,Spotify Daily Chart Global,02-26-2026,Global
2,3,Djo,End of Beginning,741,1,4856294,-161385.0,32979053,-19009,2223880869,Spotify Daily Chart Global,02-26-2026,Global
3,4,Taylor Swift,The Fate of Ophelia,147,1,4545484,20750.0,30757410,104091,1067922884,Spotify Daily Chart Global,02-26-2026,Global
4,5,Bad Bunny,BAILE INoLVIDABLE,418,2,4536650,4256.0,33278386,-751104,1273769628,Spotify Daily Chart Global,02-26-2026,Global
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,Đen,Đi Về Nhà (w/ JustaTee),1660,1,21236,-1850.0,193679,-33738,37789777,Spotify Daily Chart Vietnam,02-26-2026,Vietnam
196,197,Dhruv,double take,1472,7,21082,334.0,41830,21082,32031237,Spotify Daily Chart Vietnam,02-26-2026,Vietnam
197,198,Vũ Cát Tường,Từng Là,678,2,21032,NaN,21032,693,38007812,Spotify Daily Chart Vietnam,02-26-2026,Vietnam
198,199,MONO,Chăm Hoa,483,6,21016,NaN,103427,2105,24670401,Spotify Daily Chart Vietnam,02-26-2026,Vietnam


In [314]:
spotify_weekly_get_country = []
for entry in spotify_weekly_streaming_charts['chart_name']:
    new_entry = re.findall(r'(?<=Chart )(\w.*\w)', entry)
    spotify_weekly_get_country.append(new_entry[0])
spotify_weekly_streaming_charts['country'] = spotify_weekly_get_country

In [315]:
spotify_weekly_streaming_charts

,rank,artist,song_name,peak,weeks,Streams,Streams+,Total,chart_name,date,country
0,1,Bad Bunny,DtMF,1,60,45722046,-14850248.0,1549114511,Spotify Weekly Chart Global,02-26-2026,Global
1,2,Bad Bunny,BAILE INoLVIDABLE,2,60,33278386,-10608661.0,1273769628,Spotify Weekly Chart Global,02-26-2026,Global
2,3,Djo,End of Beginning,1,106,32979053,-2245987.0,2225140553,Spotify Weekly Chart Global,02-26-2026,Global
3,4,Bad Bunny,NUEVAYoL,3,60,32180613,-10955089.0,1040380429,Spotify Weekly Chart Global,02-26-2026,Global
4,5,Olivia Dean,Man I Need,3,27,31180371,-1623349.0,792309416,Spotify Weekly Chart Global,02-26-2026,Global
...,...,...,...,...,...,...,...,...,...,...,...
195,196,Changg,Em Không Hiểu (w/ Minh Huy),27,237,138341,-10117.0,32312825,Spotify Weekly Chart Vietnam,02-26-2026,Vietnam
196,197,Lil Zpoet,Yêu Từ Đâu Mà Ra,150,20,137045,-1849.0,3332279,Spotify Weekly Chart Vietnam,02-26-2026,Vietnam
197,198,Sơn Tùng M-TP,Chúng Ta Không Thuộc Về Nhau,60,60,137040,NaN,8636859,Spotify Weekly Chart Vietnam,02-26-2026,Vietnam
198,199,AMEE,MỘNG YU (w/ RPT MCK),3,76,136984,NaN,25553883,Spotify Weekly Chart Vietnam,02-26-2026,Vietnam


## Additional Days

In [270]:
spotify_daily_links_Feb27, spotify_weekly_links_Feb27 = get_spotify_streaming_links(spotify_url)

In [271]:
spotify_daily_streaming_charts_Feb27 = get_streaming_charts_spotify(spotify_daily_links_Feb27)

In [273]:
spotify_daily_Feb27 = clean_spotify(spotify_daily_streaming_charts_Feb27)

In [274]:
spotify_daily_Feb27

,rank,artist,song_name,Days,peak,Streams,Streams+,7Day,7Day+,Total,country,chart_name,date
0,1,Bad Bunny,DtMF,419,1,6390731,208089.0,44797128,-924918,1555505242,Global,Spotify Daily Chart Global,02-27-2026
1,2,PinkPantheress,Stateside + Zara Larsson (w/ Zara Larsson),52,2,5879944,649186.0,28982750,3305409,127988355,Global,Spotify Daily Chart Global,02-27-2026
2,3,Bruno Mars,Risk It All,1,3,5701450,NaN,5701450,5701450,5701450,Global,Spotify Daily Chart Global,02-27-2026
3,4,Bruno Mars,I Just Might,50,3,4874029,2082513.0,21056205,2050724,183179032,Global,Spotify Daily Chart Global,02-27-2026
4,5,Olivia Dean,Man I Need,192,2,4742910,242959.0,31225632,45261,799812395,Global,Spotify Daily Chart Global,02-27-2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,Da LAB,Nước Mắt Em Lau Bằng Tình Yêu Mới (w/ Tóc Tiên),699,53,20882,-7958.0,138125,-672,13254463,Vietnam,Spotify Daily Chart Vietnam,02-27-2026
196,197,Changg,Em Không Hiểu (w/ Minh Huy),1622,25,20870,-2703.0,107442,20870,31521558,Vietnam,Spotify Daily Chart Vietnam,02-27-2026
197,198,Andiez,1 Phút,1397,11,20830,NaN,20830,20830,9648562,Vietnam,Spotify Daily Chart Vietnam,02-27-2026
198,199,Dangrangto,thế giới của anh (w/ DONAL),29,24,20739,-1743.0,194268,-5778,1080092,Vietnam,Spotify Daily Chart Vietnam,02-27-2026


In [275]:
apple_daily_links_Feb28 = get_apple_streaming_links(apple_url)

In [276]:
apple_charts_28 = get_charts_apple(apple_daily_links_Feb28)

In [279]:
apple_charts_28_clean = clean_apple_charts(apple_charts_28)

In [281]:
apple_charts_28_clean

,rank,artist,song_name,country,chart_name,date
0,1,Bad Bunny,DtMF,Worldwide,Worldwide Apple Music Song,02-28-2026
1,2,Taylor Swift,The Fate of Ophelia,Worldwide,Worldwide Apple Music Song,02-28-2026
2,3,PinkPantheress,Stateside,Worldwide,Worldwide Apple Music Song,02-28-2026
3,4,Bruno Mars,Risk It All,Worldwide,Worldwide Apple Music Song,02-28-2026
4,5,Dave & Tems,Raindance,Worldwide,Worldwide Apple Music Song,02-28-2026
...,...,...,...,...,...,...
195,196,Holy Ten,Banga (feat. Kimberley Richard),Zimbabwe,Zimbabwe Apple Music Top Song,02-28-2026
196,197,Jah Prayzah,Chiringiro,Zimbabwe,Zimbabwe Apple Music Top Song,02-28-2026
197,198,Jah Prayzah,Mandionei,Zimbabwe,Zimbabwe Apple Music Top Song,02-28-2026
198,199,Saintfloew,Cheka,Zimbabwe,Zimbabwe Apple Music Top Song,02-28-2026


# Database

### SQL Database

In [316]:
#Save as sql file
# Import/Create DB from CSV
sqlite_db_path = '../all_data/data.db'  # Name of the table to be created in SQLite
conn = sql.connect(sqlite_db_path) # connect to a NEW database!

In [317]:
#Create Original charts
table_name_charts = 'BillboardCharts'
table_name_unique_artist_mentions = "IndividualArtistMentions"
table_name_all_artist_appearances = "SongArtistMentions"
table_name_unique_artist_chart_appearances = "SongArtistChartAppearances"
table_name_spotify_daily_streaming_charts = "SpotifyDailyStreamingCharts"
table_name_spotify_weekly_streaming_charts = "SpotifyWeeklyStreamingCharts"
table_name_apple_chart_rankings = "AppleChartRankings"


all_charts_df.to_sql(name=table_name_charts, con = conn, if_exists='replace', index=False)
unique_artist_mentions.to_sql(name=table_name_unique_artist_mentions, con = conn, if_exists='replace', index=False)
total_appearances_all_charts.to_sql(name=table_name_all_artist_appearances, con = conn, if_exists='replace', index=False)
unique_chart_appearances.to_sql(name=table_name_unique_artist_chart_appearances, con = conn, if_exists='replace', index=False)
spotify_daily_streaming_charts.to_sql(name=table_name_spotify_daily_streaming_charts, con = conn, if_exists='replace', index=False)
spotify_weekly_streaming_charts.to_sql(name=table_name_spotify_weekly_streaming_charts, con = conn, if_exists='replace', index=False)
apple_charts.to_sql(name=table_name_apple_chart_rankings, con = conn, if_exists='replace', index=False)

31400

In [318]:
pd.read_sql('''SELECT * from AppleChartRankings''', conn)

,rank,artist,song_name,chart_name,date,country
0,1,Bad Bunny,DtMF,Worldwide Apple Music Song,02-27-2026,Worldwide
1,2,Taylor Swift,The Fate of Ophelia,Worldwide Apple Music Song,02-27-2026,Worldwide
2,3,PinkPantheress,Stateside,Worldwide Apple Music Song,02-27-2026,Worldwide
3,4,Dave & Tems,Raindance,Worldwide Apple Music Song,02-27-2026,Worldwide
4,5,Olivia Dean,Man I Need,Worldwide Apple Music Song,02-27-2026,Worldwide
...,...,...,...,...,...,...
31395,196,Jnr Spragga,Zvakufaya (feat. Nisha Ts),Zimbabwe Apple Music Top Song,02-27-2026,Zimbabwe
31396,197,Jah Prayzah,Shuga,Zimbabwe Apple Music Top Song,02-27-2026,Zimbabwe
31397,198,"Mr Pilato, Ego Slimflow & Tebogo G Mashego","Biri Marung (feat. Sje Konka, Focalistic, DJ M...",Zimbabwe Apple Music Top Song,02-27-2026,Zimbabwe
31398,199,The Silent Partner,Im Learning (feat. IVEY.H),Zimbabwe Apple Music Top Song,02-27-2026,Zimbabwe


### Additional Days DB

Note: If we want to scrape more date from additional days, we should append to the charts above, not replace the original data.

In [140]:
def append_charts(table_name, df, conn):
    df.to_sql(name = table_name, con = conn, if_exists='append', index=False)

#### Feb 28

In [319]:
#Feb 28 Data
append_charts("SpotifyDailyStreamingCharts", spotify_daily_Feb27, conn)
append_charts("AppleChartRankings", apple_charts_28_clean, conn)

In [322]:
pd.read_sql('''SELECT * from AppleChartRankings''', conn)

,rank,artist,song_name,chart_name,date,country
0,1,Bad Bunny,DtMF,Worldwide Apple Music Song,02-27-2026,Worldwide
1,2,Taylor Swift,The Fate of Ophelia,Worldwide Apple Music Song,02-27-2026,Worldwide
2,3,PinkPantheress,Stateside,Worldwide Apple Music Song,02-27-2026,Worldwide
3,4,Dave & Tems,Raindance,Worldwide Apple Music Song,02-27-2026,Worldwide
4,5,Olivia Dean,Man I Need,Worldwide Apple Music Song,02-27-2026,Worldwide
...,...,...,...,...,...,...
62795,196,Holy Ten,Banga (feat. Kimberley Richard),Zimbabwe Apple Music Top Song,02-28-2026,Zimbabwe
62796,197,Jah Prayzah,Chiringiro,Zimbabwe Apple Music Top Song,02-28-2026,Zimbabwe
62797,198,Jah Prayzah,Mandionei,Zimbabwe Apple Music Top Song,02-28-2026,Zimbabwe
62798,199,Saintfloew,Cheka,Zimbabwe Apple Music Top Song,02-28-2026,Zimbabwe


### CSV Files


In [142]:
csv_charts_path = '../../Final_Project_Data/BillboardCharts.csv'
csv_unique_artist_mentions_path = "../../Final_Project_Data/IndividualArtistMentions.csv"
csv_all_artist_appearances_path = "../../Final_Project_Data/SongArtistAppearances.csv"
csv_unique_artist_chart_appearances_path = "../../Final_Project_Data/SongArtistChartAppearances.csv"
csv_spotify_daily_streaming_charts_path = "../../Final_Project_Data/SpotifyDailyStreamingCharts.csv"
csv_spotify_weekly_streaming_charts_path = "../../Final_Project_Data/SpotifyWeeklyStreamingCharts.csv"
csv_apple_chart_rankings_path = "../../Final_Project_Data/AppleChartRankings.csv"

all_charts_df.to_csv(csv_charts_path, index=False)
unique_artist_mentions.to_csv(csv_unique_artist_mentions_path, index=False)
total_appearances_all_charts.to_csv(csv_all_artist_appearances_path, index=False)
unique_chart_appearances.to_csv(csv_unique_artist_chart_appearances_path, index=False)
spotify_daily_streaming_charts.to_csv(csv_spotify_daily_streaming_charts_path, index=False)
spotify_weekly_streaming_charts.to_csv(csv_spotify_weekly_streaming_charts_path, index=False)
apple_charts.to_csv(csv_apple_chart_rankings_path, index=False)

In [325]:
spotify_daily_streaming_charts.to_csv(csv_spotify_daily_streaming_charts_path, index=False)
apple_charts.to_csv(csv_apple_chart_rankings_path, index=False)

In [327]:
#Feb 28 Data
csv_spotify_daily_streaming_charts_path_feb28 = "../../Final_Project_Data/SpotifyDailyStreamingCharts_0228.csv"
csv_apple_chart_rankings_path_feb28 = "../../Final_Project_Data/AppleChartRankings_0228.csv"

spotify_daily_Feb27.to_csv(csv_spotify_daily_streaming_charts_path_feb28, index=False)
apple_charts_28_clean.to_csv(csv_apple_chart_rankings_path_feb28, index=False)